In [ ]:
from pathlib import Path
import xarray as xr
import utils
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, MinMaxScaler, PolynomialFeatures
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import umap
from sklearn.datasets import load_digits
import re
import json
import pickle
from matplotlib.gridspec import GridSpec
import cartopy.crs as ccrs
import cartopy.feature as CFeature
import random
from scipy.stats import bootstrap
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.pipeline import make_pipeline
import matplotlib.colors as mcolors
import add_best_track
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
import plotly.express as px
# import tensorflow as tf
from sklearn.decomposition import IncrementalPCA
# from statsmodels.multivariate.factor_rotation import rotate_factors
from sklearn.cluster import DBSCAN
import plotly.express as px
import joblib
import TC_Embedding
import TC_EmbeddingStormAnalysis

# tf.config.run_functions_eagerly(True)



## Start in code cell 7 if CombinedDataset.nc is already created.


Some Additional Filtering that wasn't included in the original data consolidation workflow

In [ ]:
def TC_svr_classifier(feature, target):

        if len(np.shape(feature)) == 1:
            feature = feature.reshape(-1,1)
        X_train, X_test, y_train, y_test = train_test_split(feature, target)

        scaler_X = StandardScaler()
        X_train_scaled = scaler_X.fit_transform(X_train)
        X_test_scaled = scaler_X.transform(X_test)
        svr = SVR(kernel = 'rbf', C = 1.0, epsilon = 0.1)
        fitted_model = svr.fit(X_train_scaled, y_train)
        y_pred = fitted_model.predict(X_test_scaled)
        resids = y_pred - y_test
        standard_error = np.std(resids)/np.sqrt(len(y_pred))

        return {'RMSE': np.sqrt(mean_squared_error(y_test,y_pred)), 
                'Coefficient of Determination': fitted_model.score(X_test, y_test),
                'Standard Error': standard_error}

def TC_linear_classifier(feature, target):

        if len(np.shape(feature)) == 1:
            feature = feature.reshape(-1,1)
        
        X_train, X_test, y_train, y_test = train_test_split(feature, target)
        poly_model = make_pipeline(LinearRegression())
        fitted_model = poly_model.fit(X_train, y_train)
        y_pred = fitted_model.predict(X_test)
        resids = y_pred - y_test
        standard_error = np.std(resids)/np.sqrt(len(y_pred))

        return {'RMSE': np.sqrt(mean_squared_error(y_test,y_pred)), 
                'Coefficient of Determination': fitted_model.score(X_test, y_test),
                'Standard Error': standard_error}

def TC_rf_classifier(feature, target):

        if len(np.shape(feature)) == 1:
            feature = feature.reshape(-1,1)
        
        X_train, X_test, y_train, y_test = train_test_split(feature, target)
        rf_model = RandomForestRegressor(n_estimators=100, max_depth=15, n_jobs=-1, random_state=42)
        fitted_model = rf_model.fit(X_train, y_train)
        y_pred = fitted_model.predict(X_test)
        resids = y_pred - y_test
        standard_error = np.std(resids)/np.sqrt(len(y_pred))

        return {'RMSE': np.sqrt(mean_squared_error(y_test,y_pred)), 
                'Coefficient of Determination': fitted_model.score(X_test, y_test),
                'Standard Error': standard_error}

In [ ]:

# Loading and combining the 3 TC Field datasets

ds1 = xr.open_dataset('/data/keeling/a/ianpark3/a597/ThesisTCdataset/Semester2_TC_Fields_pt1.nc')
ds2 = xr.open_dataset('/data/keeling/a/ianpark3/a597/ThesisTCdataset/Semester2_TC_Fields_pt2.nc')
ds3 = xr.open_dataset('/data/keeling/a/ianpark3/a597/ThesisTCdataset/Semester2_TC_Fields_pt3.nc')

ds = xr.concat((ds1, ds2, ds3), dim = 'frame_number')


In [ ]:
ds

In [ ]:
ds_best_track = add_best_track.add_best_track(ds, 'Best_Track_Data/ibtracs.ALL.list.v04r01.csv')

In [ ]:
# Checking for NA's
ds_cleaned = ds_best_track.dropna(dim = 'frame_number', how = 'any')

#Selecting only hfsa runs
# hfsa_index = ['hfsa' in link for link in ds_cleaned['frame_link'].values]
# ds_cleaned = ds_cleaned.sel(frame_number = hfsa_index, drop = True)
# Limiting frames to tropical storm strength or stronger (17 m/s or stronger)
ds_cleaned = ds_cleaned.where(ds_cleaned['max_wind'] >= 17, drop = True)

# Limiting frames to those where the model minimum central pressure is within 4mb of best track
ds_cleaned = ds_cleaned.where((ds_cleaned['center_pressure']- ds_cleaned['best_track_pres'])<=4, drop = True)

# Only using storms in the Northern Hemisphere
ds_final = ds_cleaned.where(ds_cleaned['center_lat'] > 0, drop = True)

# ds_cleaned = ds_cleaned.dropna(dim = 'frame_number', how = 'any')


In [ ]:
ds_final

In [ ]:
fl_wind_max = np.max(np.sqrt(ds_final['rad_wind'].isel(levels = 1).values ** 2 + ds_final['az_wind'].isel(levels = 1).values ** 2), axis = (1,2))
wind_change = ds_final['max_wind_24hrs'].isel(levels = 0).values - ds_final['max_wind'].isel(levels = 0).values
# sfc_winds = ds_final['max_wind'].isel(levels = 0).values
# best_track_wind = ds_final['best_track_winds'].values[:,0]
# best_track_windchange = ds_final['best_track_24winds'].isel(levels = 0).values - ds_final['best_track_winds'].isel(levels = 0).values
# best_track_pres = ds_final['best_track_pres'].values[:,0]

# storm_category = []

# for wind in ds_final['max_wind'].isel(levels = 0).values:
#         if   wind < 33:
#                 storm_category.append(0)
#         elif 33 <= wind < 43:
#                 storm_category.append(1)
#         elif 43 <= wind < 50:
#                 storm_category.append(2)
#         elif 50 <= wind < 58:
#                 storm_category.append(3)
#         elif 58 <= wind < 70:
#                 storm_category.append(4)
#         elif 70 <= wind:
#                 storm_category.append(5)
# storm_category = np.array(storm_category)

TC_baseline = TC_rf_classifier(np.array([ds_final['center_pressure'].isel(levels = 0).values, fl_wind_max]).T, ds_final['max_wind'].isel(levels = 0).values)



In [ ]:
# Global Neighbors variable for UMAP embeddings.
NEIGHBORS = 5
MIN_DIST = 0.15
N_COMPONENTS = 2

# Setting up a custom cmap for wind change.
wind_change_bins = [-30,-14,-7, 0, 7, 14,50]
wind_change_colors = [mcolors.CSS4_COLORS.get('red'),
                      mcolors.CSS4_COLORS.get('lightcoral'),
                      mcolors.CSS4_COLORS.get('mistyrose'), 
                      mcolors.CSS4_COLORS.get('lightsteelblue'), 
                      mcolors.CSS4_COLORS.get('royalblue'),
                      mcolors.CSS4_COLORS.get('navy')]
WIND_CHANGE_CMAP = mcolors.ListedColormap(wind_change_colors)
WIND_CHANGE_NORM = mcolors.BoundaryNorm(boundaries=wind_change_bins, ncolors=len(wind_change_colors))


## Running different Embeddings

In [ ]:
fields = ['gh', 'rad_wind', 'az_wind', 't']
embed_4 = TC_Embedding.TC_embedding(ds_final, fields)
embed_4.Normalize_data()
result_4 = embed_4.Fit_umap(neighbors = NEIGHBORS, metric = 'correlation', min_dist = MIN_DIST)
result_4_lc = TC_linear_classifier(result_4, ds_final['max_wind'].isel(levels = 0).values)

In [ ]:
joblib.dump(embed_4.embedder, 'Semester2TCembedding.joblib')

In [ ]:
fields = ['gh', 'rad_wind', 'az_wind', 't']
embed_4 = TC_embedding(ds_final, fields)
embed_4.Normalize_data()
result_4 = embed_4.Fit_pumap(neighbors = NEIGHBORS, n_components = N_COMPONENTS)
result_4_lc = TC_linear_classifier(result_4, ds_final['max_wind'].isel(levels = 0).values)

In [ ]:
fields = ['gh']
embed_gh = TC_Embedding.TC_embedding(ds_final, fields)
embed_gh.Normalize_data()
result_gh = embed_gh.Fit_umap(neighbors = NEIGHBORS, metric = 'correlation', min_dist = MIN_DIST)
result_gh_lc = TC_linear_classifier(result_gh, ds_final['max_wind'].isel(levels = 0).values)

In [ ]:
fields = ['rad_wind', 'az_wind']
embed_winds =  TC_Embedding.TC_embedding(ds_final, fields)
embed_winds.Normalize_data()
result_winds = embed_winds.Fit_umap(neighbors = NEIGHBORS, metric = 'correlation', min_dist = MIN_DIST)
result_winds_lc = TC_linear_classifier(result_winds, ds_final['max_wind'].isel(levels = 0).values)

In [ ]:
fields = ['gh','rad_wind', 'az_wind']
embed_winds_gh =  TC_Embedding.TC_embedding(ds_final, fields)
embed_winds_gh.Normalize_data()
result_winds_gh = embed_winds_gh.Fit_umap(neighbors = NEIGHBORS, metric = 'correlation', min_dist = MIN_DIST)
result_winds_gh_lc = TC_linear_classifier(result_winds_gh, ds_final['max_wind'].isel(levels = 0).values)

In [ ]:
fields = ['t','rad_wind', 'az_wind']
embed_winds_t =  TC_Embedding.TC_embedding(ds_final, fields)
embed_winds_t.Normalize_data()
result_winds_t = embed_winds_t.Fit_umap(neighbors = NEIGHBORS, metric = 'correlation', min_dist = MIN_DIST)
result_winds_t_lc = TC_linear_classifier(result_winds_t, ds_final['max_wind'].isel(levels = 0).values)

In [ ]:
embed_winds_t.plot_embedding()

## Grading UMAP Information Retention

In [ ]:
def check_IR(embedding, metric):
    '''
        Takes an index embedding and associated target(eg mslp or max sfc wind) and 
        checks for the ability of a random forest tree to predict storm target based on indices alone
    '''

    # Splitting the data into test and train sets

    X_train, X_test, Y_train, Y_test = train_test_split(embedding, metric, test_size=0.25, random_state = 42)

    param_grids = [
        
    ]


## Comparing UMAP to PCA and naive index

Pulling the data from embed_4 embedding

In [ ]:
num_fields = len(embed_4.normal_data.keys())
dataset_size = len(embed_4.dataset['frame_number'].values)
array = []
first = True
for field in embed_4.normal_data.keys():
        if first:
            array = embed_4.normal_data[field].reshape(dataset_size, 720 * 501)
            first = False
        else:
            array = np.concat((array, embed_4.normal_data[field].reshape(dataset_size, 720 * 501)), axis = 1)

Constructing a simple PCA

In [ ]:
pca = PCA(n_components=100).fit_transform(array)


Plotting

In [ ]:
fig = plt.figure(figsize = (18,12), layout = 'constrained')
subfigs = fig.subfigures(nrows=4, ncols=1)
axs_row1 = subfigs[0].subplots(nrows = 1, ncols = 2)
axs_row2 = subfigs[1].subplots(nrows = 1, ncols = 2)
axs_row3 = subfigs[2].subplots(nrows = 1, ncols = 2)
axs_row4 = subfigs[3].subplots(nrows = 1, ncols = 2)


im_wind = axs_row1[0].scatter(embed_4.embedding[:,0], embed_4.embedding[:,1], 
                            c = embed_4.BestTrack_max_sfcW, cmap = 'Spectral',
                            vmin = np.min(embed_4.BestTrack_max_sfcW), vmax = np.max(embed_4.BestTrack_max_sfcW))
axs_row1[0].set_title('a)', loc = 'left')

im_windchange = axs_row1[1].scatter(embed_4.embedding[:,0], embed_4.embedding[:,1], 
                                    c = embed_4.BestTrack_24hr_windchange, cmap = WIND_CHANGE_CMAP, norm = WIND_CHANGE_NORM)
axs_row1[1].set_title('b)', loc = 'left')
subfigs[0].suptitle('GH, Temperature, and Winds')



axs_row2[0].scatter(embed_winds_gh.embedding[:,0], embed_winds_gh.embedding[:,1], 
                            c = embed_winds_gh.max_winds,  cmap = 'Spectral',
                            vmin = np.min(embed_winds_gh.BestTrack_max_sfcW), vmax = np.max(embed_winds_gh.BestTrack_max_sfcW))
axs_row2[0].set_title('c)', loc = 'left')
axs_row2[1].scatter(embed_winds_gh.embedding[:,0], embed_winds_gh.embedding[:,1],
                                    c = embed_winds_gh.wind_change, cmap = WIND_CHANGE_CMAP, 
                                    norm = WIND_CHANGE_NORM)# vmin = np.min(embed_4.wind_change), vmax = np.max(embed_4.wind_change))
axs_row2[1].set_title('d)', loc = 'left')
subfigs[1].suptitle('GH and Winds')

axs_row3[0].scatter(embed_winds_t.embedding[:,0], embed_winds_t.embedding[:,1], 
                            c = embed_winds_t.max_winds,  cmap = 'Spectral',
                            vmin = np.min(embed_winds_t.BestTrack_max_sfcW), vmax = np.max(embed_winds_t.BestTrack_max_sfcW))
axs_row3[0].set_title('e)', loc = 'left')
axs_row3[0].set_ylabel('Maximum Flight Level Wind (m/s)')
axs_row3[0].set_xlabel('Minimum Central Pressure (mb)')
axs_row3[1].scatter(embed_winds_t.embedding[:,0],  embed_winds_t.embedding[:,1],
                                    c = embed_winds_t.wind_change, cmap = WIND_CHANGE_CMAP, 
                                    norm = WIND_CHANGE_NORM)# vmin = np.min(embed_4.wind_change), vmax = np.max(embed_4.wind_change))
axs_row3[1].set_title('f)', loc = 'left')
axs_row3[1].set_xlabel('Minimum Central Pressure (mb)')
subfigs[2].suptitle('Winds and Temperature')

axs_row4[0].scatter(embed_gh.embedding[:,0], embed_gh.embedding[:,1], 
                            c = embed_gh.max_winds,  cmap = 'Spectral',
                            vmin = np.min(embed_gh.BestTrack_max_sfcW), vmax = np.max(embed_gh.BestTrack_max_sfcW))
axs_row4[0].set_title('g)', loc = 'left')
axs_row4[0].set_ylabel('Maximum Flight Level Wind (m/s)')
axs_row4[0].set_xlabel('Minimum Central Pressure (mb)')
axs_row4[1].scatter(embed_gh.embedding[:,0],  embed_gh.embedding[:,1],
                                    c = embed_gh.wind_change, cmap = WIND_CHANGE_CMAP, 
                                    norm = WIND_CHANGE_NORM)# vmin = np.min(embed_4.wind_change), vmax = np.max(embed_4.wind_change))
axs_row4[1].set_title('h)', loc = 'left')
axs_row4[1].set_xlabel('Minimum Central Pressure (mb)')
subfigs[3].suptitle('GH')

fig.colorbar(im_windchange, ax = axs_row4[1], location = "bottom", label = "24 hour Windspeed Change (m/s)")
fig.colorbar(im_wind, ax = axs_row4[0], location = "bottom", label = "Surface Max Windspeed (m/s)")
fig.suptitle('Tropical Cyclone Indices Constructed from Various 700mb Fields')

## Constructing clusters with DBSCAN

In [ ]:
embedding = embed_winds_t.embedding
clusters = DBSCAN(eps = 0.3).fit(embedding)

In [ ]:

fig = plt.figure(figsize = (15,6), layout = 'constrained')
subfigs = fig.subfigures(nrows=1, ncols=1)
axs_row1 = subfigs.subplots(nrows = 1, ncols = 2)



im_wind = axs_row1[0].scatter(embed_winds_t.embedding[:,0], embed_winds_t.embedding[:,1], 
                              c = clusters.labels_, cmap = 'rainbow'
                              )
axs_row1[0].set_title('a) DBSCAN cluster results', loc = 'left')


im_windchange = axs_row1[1].scatter(embed_winds_t.embedding[:,0], embed_winds_t.embedding[:,1], 
                                    c = embed_winds_t.BestTrack_24hr_windchange, cmap = WIND_CHANGE_CMAP, norm = WIND_CHANGE_NORM)
axs_row1[1].set_title('b) Best Track 24 hour intensity change', loc = 'left')
subfigs.suptitle('Cluster Identification')
fig.colorbar(im_wind, ax = axs_row1[0], label = 'Cluster Number')
fig.colorbar(im_windchange, ax = axs_row1[1], label = '24 Hour Change in Maximum Windspeed (m/s)')



In [ ]:
px.scatter(x = embed_winds_t.embedding[:,0], y = embed_winds_t.embedding[:,1],
           color = clusters.labels_,
           width = 800,
           height = 500
           )

Examining Specific Clusters

In [ ]:
ds

In [ ]:
embed_winds_t.plot_composites(cluster_filter=(clusters.labels_ == 20))

In [ ]:
embed_4.plot_embedding()

In [ ]:
embed_4.plot_composites(x_low=1,x_high=3,y_low = -6,y_high=-4)

In [ ]:
embed_winds.plot_embedding()

In [ ]:
embed_gh.plot_embedding()

In [ ]:
np.sum(embed_4.BestTrack_24hr_windchange >= 14)

In [ ]:
filter_y = (embed_4.embedding[:,1] > 0) & (embed_4.embedding[:,1] < 2)
filter_x = (embed_4.embedding[:,0] > 8) & (embed_4.embedding[:,0] < 10)
cluster_filter = filter_x * filter_y

fig, axs = plt.subplots(nrows = 2, ncols = 1, subplot_kw= {'projection': ccrs.PlateCarree()})

axs[0].scatter(embed_4.center_lon[cluster_filter], embed_4.center_lat[cluster_filter], transform = ccrs.PlateCarree(),
               c = embed_4.BestTrack_24hr_windchange[cluster_filter], cmap = WIND_CHANGE_CMAP, norm = WIND_CHANGE_NORM)
axs[0].add_feature(CFeature.COASTLINE)

im = axs[1].scatter(embed_4.embedding[cluster_filter,0], embed_4.embedding[cluster_filter,1], 
                    c = embed_4.BestTrack_24hr_windchange[cluster_filter], cmap = WIND_CHANGE_CMAP, norm = WIND_CHANGE_NORM)
axs[1].scatter(embed_4.embedding[~cluster_filter,0], embed_4.embedding[~cluster_filter,1], color = 'black')

fig.tight_layout()

fig.suptitle(f'Average 24 hour best track windspeed change for highlighted cluster {np.round(np.mean(embed_4.BestTrack_24hr_windchange[cluster_filter]),2)}')

In [ ]:
fig = plt.figure(figsize = (16,12), layout = 'constrained')
subfigs = fig.subfigures(nrows=2, ncols=1)
axs_row1 = subfigs[0].subplots(nrows = 1, ncols = 2)
axs_row2 = subfigs[1].subplots(nrows = 1, ncols = 2)


im_wind = axs_row1[0].scatter(embed_winds_gh.embedding[:,0], embed_winds_gh.embedding[:,1], c = embed_winds_gh.BestTrack_max_sfcW, cmap = 'Spectral')
im_windchange = axs_row1[1].scatter(embed_winds_gh.embedding[:,0], embed_winds_gh.embedding[:,1], c = embed_winds_gh.BestTrack_24hr_windchange, cmap = wind_change_cmap, norm = wind_change_norm)
subfigs[0].suptitle('Shading: Best Track Data')

im_wind = axs_row2[0].scatter(embed_4.embedding[:,0], embed_4.embedding[:,1], 
                              c = embed_4.max_winds,  cmap = 'Spectral',
                              vmin = np.min(embed_4.BestTrack_max_sfcW), vmax = np.max(embed_4.BestTrack_max_sfcW))
im_windchange = axs_row2[1].scatter(embed_4.embedding[:,0], embed_4.embedding[:,1],
                                    c = embed_4.wind_change, cmap = wind_change_cmap, 
                                    vmin = np.min(embed_4.wind_change), vmax = np.max(embed_4.wind_change))
subfigs[1].suptitle('Shading: HAFS-A Data')

fig.colorbar(im_windchange, ax = axs_row2[1], location = "bottom", label = "24 hour Windspeed Change (m/s)")
fig.colorbar(im_wind, ax = axs_row2[0], location = "bottom", label = "Max Windspeed (m/s)")


fig.suptitle(f'''TC Embedding | fields: GH, Radial Winds, Azimuthal Winds, Temperature | neighbors: {NEIGHBORS} 
                 TC Wind Classifier RMSE: {np.round(result_4_lc['RMSE'],2)} 
                 N = {len(embed_4.embedding[:,0])}''')


fig, axs = plt.subplots(nrows = 1, ncols = 2, figsize = (16,8))
im_wind = axs[0].scatter(embed_winds_gh.embedding[:,0], embed_winds_gh.embedding[:,1], c = embed_winds_gh.BestTrack_max_sfcW, cmap = 'Spectral')
im_windchange = axs[1].scatter(embed_winds_gh.embedding[:,0], embed_winds_gh.embedding[:,1],  c = embed_winds_gh.BestTrack_24hr_windchange, cmap = wind_change_cmap, norm = wind_change_norm)
fig.colorbar(im_windchange, ax = axs[1], location = "bottom", label = "24 hour Windspeed Change (m/s)")
fig.colorbar(im_wind, ax = axs[0], location = "bottom", label = "Max Windspeed (m/s)")
fig.suptitle(f'''TC Embedding | fields: GH, Radial Winds, Azimuthal Winds | neighbors: {NEIGHBORS} 
                 TC Wind Classifier RMSE: {np.round(result_winds_gh_lc['RMSE'],2)} 
                 Shading: HAFS-A Data
                 N = {len(embed_winds_gh.embedding[:,0])}''')


In [ ]:
fig, axs = plt.subplots(nrows = 1, ncols = 2, figsize = (16,8))
im_wind = axs[0].scatter(embed_winds.embedding[:,0], embed_winds.embedding[:,1], c = embed_winds.BestTrack_max_sfcW, cmap = 'Spectral')
im_windchange = axs[1].scatter(embed_winds.embedding[:,0], embed_winds.embedding[:,1],  c = embed_winds_gh.BestTrack_24hr_windchange, cmap = wind_change_cmap, norm = wind_change_norm)
fig.colorbar(im_windchange, ax = axs[1], location = "bottom", label = "24 hour Windspeed Change (m/s)")
fig.colorbar(im_wind, ax = axs[0], location = "bottom", label = "Max Windspeed (m/s)")

fig.suptitle(f'''TC Embedding | fields: Radial Winds, Azimuthal Winds | neighbors: {NEIGHBORS} 
                 TC Wind Classifier RMSE: {np.round(result_winds_lc['RMSE'],2)} 
                 Shading: HAFS-A Data
                 N = {len(embed_4.embedding[:,0])}''')

In [ ]:
fig, axs = plt.subplots(nrows = 1, ncols = 2, figsize = (16,8))
im_wind = axs[0].scatter(embed_gh.embedding[:,0], embed_gh.embedding[:,1], c = embed_gh.BestTrack_max_sfcW, cmap = 'Spectral')
im_windchange = axs[1].scatter(embed_gh.embedding[:,0], embed_gh.embedding[:,1],  c = embed_gh.BestTrack_24hr_windchange, cmap = wind_change_cmap, norm = wind_change_norm)
fig.colorbar(im_windchange, ax = axs[1], location = "bottom", label = "24 hour Windspeed Change (m/s)")
fig.colorbar(im_wind, ax = axs[0], location = "bottom", label = "Max Windspeed (m/s)")

fig.suptitle(f'''TC Embedding | fields: GH | neighbors: {NEIGHBORS} 
                 TC Wind Classifier RMSE: {np.round(result_gh_lc['RMSE'],2)} 
                 Shading: HAFS-A Data
                 N = {len(embed_gh.embedding[:,0])}''')

In [ ]:
fig, axs = plt.subplots(nrows = 1, ncols = 2, figsize = (16,8))
im_wind = axs[0].scatter(fl_wind_max, ds_final['center_pressure'].isel(levels = 0).values, c = ds_final["max_wind"].isel(levels = 0).values, cmap = 'Spectral')
im_windchange = axs[1].scatter(fl_wind_max, ds_final['center_pressure'].isel(levels = 0).values,  c = wind_change, cmap = wind_change_cmap, norm = wind_change_norm)
fig.colorbar(im_windchange, ax = axs[1], location = "bottom", label = "24 hour Windspeed Change (m/s)")
fig.colorbar(im_wind, ax = axs[0], location = "bottom", label = "Max Windspeed (m/s)")

fig.suptitle(f'TC Baseline, fields: FL Max Wind, Minimum Central Pressure  \n TC Wind Classifier RMSE: {np.round(TC_baseline['RMSE'],2)}')

Computing Composites for two strengthening and two weakening clusters in the 700mb GPH

In [ ]:
embedding = embed_4
level = 1

# Creating the embedding filter
filter_y = (embedding.embedding[:,1] > 3) & (embedding.embedding[:,1] < 5)
filter_x = (embedding.embedding[:,0] > 2) & (embedding.embedding[:,0] < 3)
cluster_filter = filter_x * filter_y

# Creating Composite Fields
composite_filter = cluster_filter
angle = embedding.dataset.angle.values
radius = embedding.dataset.radius.values

ds_gph = embedding.dataset['gh'].isel(levels = level).values
ds_t = embedding.dataset['t'].isel(levels = level).values
ds_rad_winds = embedding.dataset['rad_wind'].isel(levels = level).values
ds_az_winds = embedding.dataset['az_wind'].isel(levels = level).values
max_winds = embedding.dataset['max_wind'].isel(levels = 0).values


cluster_wind_change = embedding.BestTrack_24hr_windchange[composite_filter]
avg_cluster_wind_change = np.nanmean(cluster_wind_change)
cluster_winds = embedding.BestTrack_max_sfcW[composite_filter]
avg_cluster_wind = np.nanmean(cluster_winds)
cluster_size =np.sum(composite_filter)
cluster_lats = embedding.center_lat[composite_filter] #0 for selecting the surface center lat
cluster_lons = embedding.center_lon[composite_filter] #0 for selecting the surface center lon

#### GH Composite ####
mean_field_gh = np.mean(ds_gph, axis = 0)
field_anomalies_gh = ds_gph - mean_field_gh
cluster_field_anomalies_gh = field_anomalies_gh[composite_filter, :, :]
cluster_field_mean_anom_gh = np.mean(cluster_field_anomalies_gh, axis = 0)
cluster_field_mean_gh = np.mean(ds_gph[composite_filter,:,:], axis = 0)

#### T Composite #####
mean_field_t = np.mean(ds_t, axis = 0)
field_anomalies_t = ds_t - mean_field_t
cluster_field_anomalies_t = field_anomalies_t[composite_filter, :, :]
cluster_field_mean_anom_t = np.mean(cluster_field_anomalies_t, axis = 0)
cluster_field_mean_t = np.mean(ds_t[composite_filter,:,:], axis = 0)

#### Rad Winds Composite #####
mean_field_rad_winds = np.mean(ds_rad_winds, axis = 0)
field_anomalies_rad_winds = ds_rad_winds - mean_field_rad_winds
cluster_field_anomalies_rad_winds = field_anomalies_rad_winds[composite_filter, :, :]
cluster_field_mean_anom_rad_winds = np.mean(cluster_field_anomalies_rad_winds, axis = 0)
cluster_field_mean_rad_winds = np.mean(ds_rad_winds[composite_filter,:,:], axis = 0)

#### Az Winds Composite #####
mean_field_az_winds = np.mean(ds_az_winds, axis = 0)
field_anomalies_az_winds = ds_az_winds - mean_field_az_winds
cluster_field_anomalies_az_winds = field_anomalies_az_winds[composite_filter, :, :]
cluster_field_mean_anom_az_winds = np.mean(cluster_field_anomalies_az_winds, axis = 0)
cluster_field_mean_az_winds = np.mean(ds_az_winds[composite_filter,:,:], axis = 0)





## Looking at Specific Storm Movement through the index

In [ ]:

ds1 = xr.open_dataset('Model_Data/CompleteData/HurricaneMelissa1To50.nc')
ds2 = xr.open_dataset('Model_Data/CompleteData/HurricaneMelissa51To74.nc')


ds = xr.concat((ds1, ds2), dim = 'valid_time')

ds_best_track = add_best_track.add_best_track(ds, 'Best_Track_Data/ibtracs.ALL.list.v04r01.csv')

ds_cleaned = ds_best_track.dropna(dim = 'frame_number', how = 'any')

hfsa_index = ['hfsa' in link for link in ds_cleaned['frame_link'].values]
ds_cleaned = ds_cleaned.sel(valid_time = hfsa_index, drop = True)

fields = ['gh', 'rad_wind', 'az_wind', 't']
embed_Melissa = TC_EmbeddingStormAnalysis.TC_embedding(ds_cleaned, fields)
embed_Melissa.Normalize_data()
# embed_ISA = TC_EmbeddingStormAnalysis.TC_embedding(ds_cleaned, fields)
# embed_ISA.Normalize_data()

In [ ]:
embed_Melissa.model = embed_4.model

In [ ]:
embed_Melissa.transform()

In [ ]:
fig = plt.figure(figsize = (16,12), )
subfigs = fig.subfigures(nrows=2, ncols=1)
axs_row1 = subfigs[0].subplots(nrows = 1, ncols = 1)



im_wind = axs_row1.scatter(embed_4.embedding[:,0], embed_4.embedding[:,1], 
                            c = embed_4.BestTrack_24hr_windchange, cmap = WIND_CHANGE_CMAP, norm = WIND_CHANGE_NORM,
                           )

fig.colorbar(im_wind, ax = axs_row1, location = "bottom", label = "24 Hour Windchange (m/s)")

axs_row1.plot(embed_Melissa.embedding[:,0], embed_Melissa.embedding[:,1], color = 'black')
axs_row1.text(embed_Melissa.embedding[0,0], embed_Melissa.embedding[0,1], "Start")
axs_row1.text(embed_Melissa.embedding[-1,0], embed_Melissa.embedding[-1,1], "End")


axs_row2 = subfigs[1].subplots(nrows = 1, ncols = 1, subplot_kw = {'projection': ccrs.PlateCarree()})
im_windchange = axs_row2.scatter(embed_Melissa.center_lon[0,:], embed_Melissa.center_lat[0,:], 
                 transform = ccrs.PlateCarree(),
                 c = embed_Melissa.BestTrack_max_sfcW[:37], cmap = 'Spectral',
                 vmin = np.min(embed_4.BestTrack_max_sfcW), vmax = np.max(embed_4.BestTrack_max_sfcW))
axs_row2.add_feature(CFeature.COASTLINE)
gl = axs_row2.gridlines(draw_labels=True, crs=ccrs.PlateCarree())

fig.colorbar(im_windchange, ax = axs_row2, location = "bottom", label = "Max Surface Wind (m/s)")

fig.suptitle(f'Hurricane Melissa: {embed_Melissa.dataset['valid_time'][0].values} to {embed_Melissa.dataset['valid_time'][-1].values}')

In [ ]:
embed_ISA.center_lat[0,:12]

In [ ]:
plt.scatter(embed_4.embedding[:,0], embed_4.embedding[:,1], alpha = 0.5)
plt.plot(embed_ISA.embedding[35:48,0], embed_ISA.embedding[35:48,1])

In [ ]:
embed_4.plot_embedding()

In [ ]:
embed_4.dataset